# Лабораторная работа №5
## Критерии согласия и однородности

**Уровень значимости:** $\alpha = 0.05$

**Нотация:** $\mathcal{N}(\mu, \sigma^2)$ — нормальное распределение со средним $\mu$ и дисперсией $\sigma^2$.

In [2]:
import numpy as np
import scipy.stats as stats

np.random.seed(42)
alpha = 0.05
n = 500

def decision(p_value, alpha=0.05):
    if p_value < alpha:
        return f"p = {p_value:.4f} < {alpha}  ->  H0 ОТВЕРГАЕТСЯ"
    return f"p = {p_value:.4f} >= {alpha}  ->  H0 НЕ ОТВЕРГАЕТСЯ"

---
## Раздел 1. Критерий Пирсона (χ²) для проверки нормальности

Разбиваем выборку на $k$ равновероятных интервалов, сравниваем наблюдаемые частоты с ожидаемыми ($n/k$). Параметры $\mu$ и $\sigma^2$ оцениваем по выборке → $df = k - 3$.

In [3]:
def pearson_normality_test(sample, k=10):
    n = len(sample)
    mu_hat = np.mean(sample)
    sigma_hat = np.std(sample, ddof=1)
    quantiles = np.linspace(0, 1, k + 1)
    bins = stats.norm.ppf(quantiles, loc=mu_hat, scale=sigma_hat)
    bins[0] = -np.inf
    bins[-1] = np.inf
    observed, _ = np.histogram(sample, bins=bins)
    expected = np.full(k, n / k)
    chi2_stat = np.sum((observed - expected) ** 2 / expected)
    df = k - 3
    p_value = 1 - stats.chi2.cdf(chi2_stat, df)
    return chi2_stat, p_value, df

### 1.1.1. $X \in \mathcal{N}(5, 9)$
Данные нормальны по построению. **Ожидаем:** $H_0$ не отвергается.

In [4]:
X_normal = np.random.normal(loc=5, scale=3, size=n)  # scale=sqrt(9)=3
chi2, p, df = pearson_normality_test(X_normal)
print(f"chi2 = {chi2:.4f},  df = {df}")
print(decision(p))

chi2 = 6.8000,  df = 7
p = 0.4500 >= 0.05  ->  H0 НЕ ОТВЕРГАЕТСЯ


### 1.1.2. $X \in U(−5, 3)$
Равномерное — плоское, не нормальное. **Ожидаем:** $H_0$ отвергается.

In [5]:
X_uniform = np.random.uniform(low=-5, high=3, size=n)
chi2, p, df = pearson_normality_test(X_uniform)
print(f"chi2 = {chi2:.4f},  df = {df}")
print(decision(p))

chi2 = 46.1200,  df = 7
p = 0.0000 < 0.05  ->  H0 ОТВЕРГАЕТСЯ


### 1.1.3. $X \in C(0, 1)$
Тяжёлые хвосты, нет математического ожидания. **Ожидаем:** $H_0$ отвергается.

In [6]:
X_cauchy = stats.cauchy.rvs(loc=0, scale=1, size=n)
chi2, p, df = pearson_normality_test(X_cauchy)
print(f"chi2 = {chi2:.4f},  df = {df}")
print(decision(p))

chi2 = 2340.4000,  df = 7
p = 0.0000 < 0.05  ->  H0 ОТВЕРГАЕТСЯ


### 1.1.4. $Y = X + 0.1·\epsilon,\text{ }X \in \mathcal{N}(5, 9),\text{ }\epsilon \in C(0, 1)$
«Загрязнённая» нормальная выборка: кошиевский шум мал (0.1), но тяжёлые хвосты присутствуют.

In [7]:
X_base = np.random.normal(loc=5, scale=3, size=n)
eps    = stats.cauchy.rvs(loc=0, scale=1, size=n)
Y_mix  = X_base + 0.1 * eps
chi2, p, df = pearson_normality_test(Y_mix)
print(f"chi2 = {chi2:.4f},  df = {df}")
print(decision(p))

chi2 = 744.8000,  df = 7
p = 0.0000 < 0.05  ->  H0 ОТВЕРГАЕТСЯ


---
## Раздел 2.1. Критерий Колмогорова для проверки нормальности

Одновыборочный K-S сравнивает ECDF с теоретической $\mathcal{N}(\hat{\mu}, \hat{\sigma}^2)$.

In [8]:
def kolmogorov_normality_test(sample):
    mu_hat    = np.mean(sample)
    sigma_hat = np.std(sample, ddof=1)
    stat, p = stats.kstest(sample, 'norm', args=(mu_hat, sigma_hat))
    return stat, p

datasets = {
    "1.1.1  N(5,9)":        X_normal,
    "1.1.2  Uniform(-5,3)": X_uniform,
    "1.1.3  Cauchy(0,1)":   X_cauchy,
    "1.1.4  Y=X+0.1eps":    Y_mix,
}
for name, sample in datasets.items():
    stat, p = kolmogorov_normality_test(sample)
    print(f"{name}:  D = {stat:.4f},  {decision(p)}")

1.1.1  N(5,9):  D = 0.0271,  p = 0.8461 >= 0.05  ->  H0 НЕ ОТВЕРГАЕТСЯ
1.1.2  Uniform(-5,3):  D = 0.0823,  p = 0.0021 < 0.05  ->  H0 ОТВЕРГАЕТСЯ
1.1.3  Cauchy(0,1):  D = 0.3809,  p = 0.0000 < 0.05  ->  H0 ОТВЕРГАЕТСЯ
1.1.4  Y=X+0.1eps:  D = 0.2883,  p = 0.0000 < 0.05  ->  H0 ОТВЕРГАЕТСЯ


---
## Раздел 2.2. Критерий Колмогорова–Смирнова для однородности

Двухвыборочный K–S проверяет $H_0$: $F_1(x) = F_2(x)$.

Базовая выборка: $X(1) \in \mathcal{N}(6, 7)$.

In [9]:
X1 = np.random.normal(loc=6, scale=np.sqrt(7), size=n)

### 2.2.1. $X(1)$ разделить пополам
Обе половины из одного распределения. **Ожидаем:** $H_0$ не отвергается.

In [10]:
X1_A, X1_B = X1[:n//2], X1[n//2:]
stat, p = stats.ks_2samp(X1_A, X1_B)
print(f"D = {stat:.4f},  {decision(p)}")

D = 0.0360,  p = 0.9970 >= 0.05  ->  H0 НЕ ОТВЕРГАЕТСЯ


### 2.2.2. $X(1) \in \mathcal{N}(6, 7)$  vs  $X(2) \in \mathcal{N}(5, 3)$
Разные $\mu$ и $\sigma^2$. **Ожидаем:** $H_0$ отвергается.

In [11]:
X2 = np.random.normal(loc=5, scale=np.sqrt(3), size=n)
stat, p = stats.ks_2samp(X1, X2)
print(f"D = {stat:.4f},  {decision(p)}")

D = 0.2840,  p = 0.0000 < 0.05  ->  H0 ОТВЕРГАЕТСЯ


### 2.2.3. $X(1) \in \mathcal{N}(6, 7)$  vs  $X(3) \in \mathcal{N}(6, 6.6)$
$\mu$ совпадает, $\sigma^2$ близко (7 vs 6.6). Различие небольшое — интересно, уловит ли критерий.

In [12]:
X3 = np.random.normal(loc=6, scale=np.sqrt(6.6), size=n)
stat, p = stats.ks_2samp(X1, X3)
print(f"D = {stat:.4f},  {decision(p)}")

D = 0.0600,  p = 0.3294 >= 0.05  ->  H0 НЕ ОТВЕРГАЕТСЯ


---
## Раздел 3. Критерий знаков для однородности параметра положения

Для двух выборок длины m формируем разности $Z_i = X_i - Y_i$.

Базовая выборка: $X(1) ∈ N(4, 7)$.

In [13]:
def sign_test(sample1, sample2):
    m = min(len(sample1), len(sample2))
    diff = sample1[:m] - sample2[:m]
    diff = diff[diff != 0]
    n_nonzero = len(diff)
    S_plus = int(np.sum(diff > 0))
    p = 2 * min(
        stats.binom.cdf(S_plus, n_nonzero, 0.5),
        1 - stats.binom.cdf(S_plus - 1, n_nonzero, 0.5)
    )
    return S_plus, n_nonzero, p

Z1 = np.random.normal(loc=4, scale=np.sqrt(7), size=n)

### 3.1.1. $X(1)$ разделить пополам
**Ожидаем:** $H_0$ не отвергается.

In [14]:
Z1_A, Z1_B = Z1[:n//2], Z1[n//2:]
S_plus, m, p = sign_test(Z1_A, Z1_B)
print(f"S+ = {S_plus},  m = {m},  {decision(p)}")

S+ = 122,  m = 250,  p = 0.7519 >= 0.05  ->  H0 НЕ ОТВЕРГАЕТСЯ


### 3.1.2. $X(1) \in \mathcal{N}(4, 7)$  vs  $X(2) \in \mathcal{N}(6, 3)$
Разница средних 4 vs 6. **Ожидаем:** $H_0$ отвергается.

In [15]:
Z2 = np.random.normal(loc=6, scale=np.sqrt(3), size=n)
S_plus, m, p = sign_test(Z1, Z2)
print(f"S+ = {S_plus},  m = {m},  {decision(p)}")

S+ = 129,  m = 500,  p = 0.0000 < 0.05  ->  H0 ОТВЕРГАЕТСЯ


### 3.1.3. $X(1) \in \mathcal{N}(4, 7)$  vs  $X(3) \in \mathcal{N}(4.5, 6)$
Малая разница средних (0.5). Критерий знаков слаб — может не уловить.

In [16]:
Z3 = np.random.normal(loc=4.5, scale=np.sqrt(6), size=n)
S_plus, m, p = sign_test(Z1, Z3)
print(f"S+ = {S_plus},  m = {m},  {decision(p)}")

S+ = 208,  m = 500,  p = 0.0002 < 0.05  ->  H0 ОТВЕРГАЕТСЯ


---
## Раздел 4. Критерий Вилкоксона (ранговых знаков)

Двухвыборочный критерий Вилкоксона (rank-sum) — более мощный аналог критерия знаков.
Учитывает не только знак разности, но и ранг её абсолютной величины.

Используем `scipy.stats.ranksums` для двух независимых выборок.

### 4 / 3.1.1. $X(1)$ разделить пополам
**Ожидаем:** $H_0$ не отвергается.

In [17]:
stat, p = stats.ranksums(Z1_A, Z1_B)
print(f"W = {stat:.4f},  {decision(p)}")

W = -1.0283,  p = 0.3038 >= 0.05  ->  H0 НЕ ОТВЕРГАЕТСЯ


### 4 / 3.1.2. $X(1) \in \mathcal{N}(4, 7)$  vs  $X(2) \in \mathcal{N}(6, 3)$
**Ожидаем:** $H_0$ отвергается, $p$-value меньше, чем у критерия знаков.

In [18]:
stat, p = stats.ranksums(Z1, Z2)
print(f"W = {stat:.4f},  {decision(p)}")

W = -13.7178,  p = 0.0000 < 0.05  ->  H0 ОТВЕРГАЕТСЯ


### 4 / 3.1.3. $X(1) \in \mathcal{N}(4, 7)$  vs  $X(3) \in \mathcal{N}(4.5, 6)$
Малая разница средних (0.5). Вилкоксон мощнее — возможно обнаружит различие там, где знаковый не смог.

In [19]:
stat, p = stats.ranksums(Z1, Z3)
print(f"W = {stat:.4f},  {decision(p)}")

W = -4.3095,  p = 0.0000 < 0.05  ->  H0 ОТВЕРГАЕТСЯ


---
## Итоговая сводная таблица

In [ ]:
print("=" * 65)
print("РАЗДЕЛ 1 & 2.1 — Проверка нормальности")
print("=" * 65)
rows = [
    ("N(5,9)",        X_normal),
    ("Uniform(-5,3)", X_uniform),
    ("Cauchy(0,1)",   X_cauchy),
    ("Y=X+0.1eps",    Y_mix),
]
print(f"{'Выборка':<18} {'Пирсон (p)':>14} {'Колмогоров (p)':>16}")
print("-" * 52)
for name, sample in rows:
    _, p_chi, _ = pearson_normality_test(sample)
    _, p_ks    = kolmogorov_normality_test(sample)
    cm = "ОТВЕРГ." if p_chi < alpha else "не отв."
    km = "ОТВЕРГ." if p_ks  < alpha else "не отв."
    print(f"{name:<18}  {p_chi:>8.4f} ({cm})  {p_ks:>8.4f} ({km})")

print()
print("=" * 65)
print("РАЗДЕЛ 2.2 — K-S однородность (X1 = N(6,7))")
print("=" * 65)
ks_pairs = [
    ("X1_A vs X1_B (половины)", X1_A, X1_B),
    ("X1 vs X2 N(5,3)",         X1,   X2),
    ("X1 vs X3 N(6,6.6)",       X1,   X3),
]
for name, a, b in ks_pairs:
    d, p = stats.ks_2samp(a, b)
    m = "ОТВЕРГ." if p < alpha else "не отв."
    print(f"{name:<30}  D={d:.4f}  p={p:.4f}  ({m})")

print()
print("=" * 65)
print("РАЗДЕЛ 3 & 4 — Однородность положения (Z1 = N(4,7))")
print("=" * 65)
sign_pairs = [
    ("Z1_A vs Z1_B (половины)", Z1_A, Z1_B),
    ("Z1 vs Z2 N(6,3)",         Z1,   Z2),
    ("Z1 vs Z3 N(4.5,6)",       Z1,   Z3),
]
print(f"{'Пара':<30} {'Знаки (p)':>12} {'Вилкоксон (p)':>15}")
print("-" * 62)
for name, a, b in sign_pairs:
    _, _, p_sign = sign_test(a, b)
    _, p_wilc    = stats.ranksums(a, b)
    sm = "ОТВЕРГ." if p_sign < alpha else "не отв."
    wm = "ОТВЕРГ." if p_wilc < alpha else "не отв."
    print(f"{name:<30}  {p_sign:>8.4f} ({sm})  {p_wilc:>8.4f} ({wm})")

РАЗДЕЛ 1 & 2.1 — Проверка нормальности
Выборка                Пирсон (p)   Колмогоров (p)
----------------------------------------------------
N(5,9)                0.4500 (не отв. )    0.8461 (не отв. )
Uniform(-5,3)         0.0000 (ОТВЕРГ.)    0.0021 (ОТВЕРГ.)
Cauchy(0,1)           0.0000 (ОТВЕРГ.)    0.0000 (ОТВЕРГ.)
Y=X+0.1eps            0.0000 (ОТВЕРГ.)    0.0000 (ОТВЕРГ.)

РАЗДЕЛ 2.2 — K-S однородность (X1 = N(6,7))
X1_A vs X1_B (половины)         D=0.0360  p=0.9970  (не отв. )
X1 vs X2 N(5,3)                 D=0.2840  p=0.0000  (ОТВЕРГ.)
X1 vs X3 N(6,6.6)               D=0.0600  p=0.3294  (не отв. )

РАЗДЕЛ 3 & 4 — Однородность положения (Z1 = N(4,7))
Пара                              Знаки (p)   Вилкоксон (p)
--------------------------------------------------------------
Z1_A vs Z1_B (половины)           0.7519 (не отв. )    0.3038 (не отв. )
Z1 vs Z2 N(6,3)                   0.0000 (ОТВЕРГ.)    0.0000 (ОТВЕРГ.)
Z1 vs Z3 N(4.5,6)                 0.0002 (ОТВЕРГ.)    0.0000 (ОТВЕ